# Stage 2 — Wider ensemble: count features, entity-embedding NN, stacking

**Question:** do more diverse models and a learned stacker help?

- CatBoost with age / household size as categories plus target-free count features (train+test frequencies).
- PyTorch entity-embedding MLP; logistic regression with three-way crosses and a tuned `C`.
- Weighted probability blend vs. a per-country logistic stacker, both judged on held-out OOF rows.

| result | CV logloss | public LB | wrong rows |
|---|---|---|---|
| **blend** | **0.27754** | **0.108765** | **1097** |
| stacker | 0.27832 | 0.109459 | 1104 |

**Finding:** CV *logloss* ranked all three submissions so far in exactly the leaderboard's order; CV *error rate* did not. Every later decision is made on held-out logloss.

**Runs on Kaggle** (GPU T4 x2, Internet on for `pip install`). Inputs and the saved-prediction chain are described in the [README](../README.md#reproducing).

In [1]:
import os, glob, time, subprocess, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

def find_file(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(name)
    return os.path.abspath(hits[0]) if hits else None

def gpu_available():
    try:
        return subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
    except FileNotFoundError:
        return False

TRAIN_PATH, TEST_PATH = find_file('Train.csv'), find_file('Test.csv')
if not TRAIN_PATH:
    raise FileNotFoundError('Train.csv not found - attach the dataset via "Add Input".')
OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
USE_GPU = gpu_available()

# ---- knobs ----------------------------------------------------------------
FAST = os.environ.get('FAST') == '1'      # quick smoke test (1 seed, few trees/epochs)
N_FOLDS = 5
SEEDS = [0] if FAST else [0, 1, 2, 3, 4]
CATBOOST_ON_GPU = True    # v1: GPU 0.11156 vs CPU ~0.11125 locally; set False to try CPU (slower, ~2x)
RUN = dict(catboost=True, catboost_v2=True, lightgbm=True, logreg=True, nn=True)
PREVIOUS_BEST = 0.109161
# ---------------------------------------------------------------------------

print('train:', TRAIN_PATH, '| out:', OUT_DIR, '| GPU:', USE_GPU, '| FAST:', FAST)

train: /kaggle/input/datasets/jeffmogaka/datadef/Train.csv | out: /kaggle/working | GPU: True | FAST: False


## Data and features

In [2]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
y = (train['bank_account'] == 'Yes').astype(int).values
test_ids = test['uniqueid'] + ' x ' + test['country']

CATS = ['country', 'location_type', 'cellphone_access', 'gender_of_respondent',
        'relationship_with_head', 'marital_status', 'education_level', 'job_type']
NUMS = ['household_size', 'age_of_respondent']   # `year` maps 1:1 to country, so it is dropped

X = train[CATS + NUMS].reset_index(drop=True)
X_test = test[CATS + NUMS].reset_index(drop=True)

# Extra features. Counts use train+test together but never the target, so they cannot leak.
both = pd.concat([X, X_test], ignore_index=True)
def count_of(cols):
    return both.groupby(cols)[cols[0]].transform('size').values

extra = pd.DataFrame({
    'age_cat': both['age_of_respondent'].astype(str),
    'hh_cat': both['household_size'].clip(1, 15).astype(str),
    'cnt_profile': count_of(CATS + NUMS),                       # identical respondents
    'cnt_ctry_edu_job': count_of(['country', 'education_level', 'job_type']),
    'cnt_ctry_loc_cell': count_of(['country', 'location_type', 'cellphone_access']),
    'cnt_ctry_age': count_of(['country', 'age_of_respondent']),
})
X2 = pd.concat([X, extra.iloc[:len(X)].reset_index(drop=True)], axis=1)
X2_test = pd.concat([X_test, extra.iloc[len(X):].reset_index(drop=True)], axis=1)
CATS2 = CATS + ['age_cat', 'hh_cat']
CNTS = ['cnt_profile', 'cnt_ctry_edu_job', 'cnt_ctry_loc_cell', 'cnt_ctry_age']

print(X2.shape, X2_test.shape, '| positive rate: %.4f' % y.mean())

(23524, 16) (10086, 16) | positive rate: 0.1408


## Cross-validation harness
Same as v1: every model sees identical folds for every seed, so OOF predictions are comparable and can be stacked.

In [3]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

FOLDS = {s: list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=s).split(X, y)) for s in SEEDS}
OOF, TEST, SCORES = {}, {}, {}

def error_rate(p, t=0.5, target=None):
    target = y if target is None else target
    return np.mean((p > t).astype(int) != target)

def run_model(name, fit_predict):
    t0 = time.time()
    oof = np.zeros((len(SEEDS), len(y)))
    tst = np.zeros(len(X_test))
    for k, s in enumerate(SEEDS):
        for tr_idx, va_idx in FOLDS[s]:
            p_va, p_te = fit_predict(tr_idx, va_idx, s)
            oof[k, va_idx] = p_va
            tst += p_te / (len(SEEDS) * N_FOLDS)
    per_seed = [error_rate(o) for o in oof]
    OOF[name], TEST[name] = oof.mean(0), tst
    SCORES[name] = dict(cv_error=np.mean(per_seed), cv_error_std=np.std(per_seed),
                        logloss=np.mean([log_loss(y, o) for o in oof]), minutes=(time.time() - t0) / 60)
    print(f"{name:14s} CV error {np.mean(per_seed):.5f} ± {np.std(per_seed):.5f} | "
          f"logloss {SCORES[name]['logloss']:.5f} | {SCORES[name]['minutes']:.1f} min")

## Model 1 — CatBoost (v1 settings, our best single model)

In [4]:
from catboost import CatBoostClassifier

def make_catboost(params, seed):
    extra = dict(task_type='GPU', devices='0') if (USE_GPU and CATBOOST_ON_GPU) else dict(thread_count=-1)
    return CatBoostClassifier(**params, **extra, random_seed=seed, verbose=0, allow_writing_files=False)

CB_PARAMS = dict(iterations=300 if FAST else 1000, learning_rate=0.03, depth=6,
                 l2_leaf_reg=5, max_ctr_complexity=2, border_count=254)

def catboost_fp(tr_idx, va_idx, seed):
    m = make_catboost(CB_PARAMS, seed)
    m.fit(X.iloc[tr_idx], y[tr_idx], cat_features=CATS)
    return m.predict_proba(X.iloc[va_idx])[:, 1], m.predict_proba(X_test)[:, 1]

if RUN['catboost']:
    run_model('catboost', catboost_fp)

catboost       CV error 0.11148 ± 0.00047 | logloss 0.27929 | 15.6 min


## Model 2 — CatBoost v2 (age/household as categories + count features)
CatBoost builds target statistics for every category, so treating exact age and household size as categories lets it learn non-smooth effects (e.g. specific ages) that trees on the raw numbers split coarsely.

In [5]:
CB2_PARAMS = dict(iterations=300 if FAST else 1200, learning_rate=0.03, depth=6,
                  l2_leaf_reg=8, max_ctr_complexity=2, border_count=254, random_strength=1.5)

def catboost_v2_fp(tr_idx, va_idx, seed):
    m = make_catboost(CB2_PARAMS, seed + 100)
    m.fit(X2.iloc[tr_idx], y[tr_idx], cat_features=CATS2)
    return m.predict_proba(X2.iloc[va_idx])[:, 1], m.predict_proba(X2_test)[:, 1]

if RUN['catboost_v2']:
    run_model('catboost_v2', catboost_v2_fp)

catboost_v2    CV error 0.11225 ± 0.00040 | logloss 0.27913 | 20.0 min


## Model 3 — LightGBM (native categoricals + counts)

In [6]:
import lightgbm as lgb

def as_category(df, ref):
    df = df.copy()
    for c in CATS:
        df[c] = pd.Categorical(df[c], categories=sorted(set(ref[0][c]) | set(ref[1][c])))
    return df

LGB_COLS = CATS + NUMS + CNTS
Xc = as_category(X2[LGB_COLS], (X2, X2_test))
Xc_test = as_category(X2_test[LGB_COLS], (X2, X2_test))

LGB_PARAMS = dict(n_estimators=150 if FAST else 700, learning_rate=0.02, num_leaves=24, min_child_samples=40,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.7, cat_smooth=20, cat_l2=10,
                  reg_lambda=3, verbose=-1, n_jobs=-1)

def lgb_fp(tr_idx, va_idx, seed):
    m = lgb.LGBMClassifier(**LGB_PARAMS, random_state=seed)
    m.fit(Xc.iloc[tr_idx], y[tr_idx])
    return m.predict_proba(Xc.iloc[va_idx])[:, 1], m.predict_proba(Xc_test)[:, 1]

if RUN['lightgbm']:
    run_model('lightgbm', lgb_fp)

lightgbm       CV error 0.11260 ± 0.00067 | logloss 0.28157 | 0.6 min


## Model 4 — Logistic regression on pairwise + three-way crosses
The regularisation strength `C` is chosen by a quick grid on one seed (by logloss) before the full run.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder

TRIPLES = [('country', 'education_level', 'job_type'),
           ('country', 'location_type', 'cellphone_access'),
           ('country', 'age_b', 'gender_of_respondent'),
           ('country', 'education_level', 'age_b'),
           ('country', 'relationship_with_head', 'marital_status')]

def crosses(df):
    d = df[CATS].copy()
    d['age_b'] = pd.cut(df['age_of_respondent'], [0, 20, 25, 30, 35, 40, 50, 60, 70, 200]).astype(str)
    d['hh_b'] = df['household_size'].clip(1, 12).astype(str)
    base = list(d.columns)
    for i in range(len(base)):
        for j in range(i + 1, len(base)):
            d[base[i] + '*' + base[j]] = d[base[i]] + '|' + d[base[j]]
    for a, b, c in TRIPLES:
        d[f'{a}*{b}*{c}'] = d[a] + '|' + d[b] + '|' + d[c]
    return d

Xx, Xx_test = crosses(X), crosses(X_test)

def make_lr_fp(C):
    def lr_fp(tr_idx, va_idx, seed):
        enc = OneHotEncoder(handle_unknown='ignore', min_frequency=5)
        A = enc.fit_transform(Xx.iloc[tr_idx])
        m = LogisticRegression(C=C, max_iter=3000)
        m.fit(A, y[tr_idx])
        return m.predict_proba(enc.transform(Xx.iloc[va_idx]))[:, 1], m.predict_proba(enc.transform(Xx_test))[:, 1]
    return lr_fp

if RUN['logreg']:
    grid = {}
    for C in [0.02, 0.035, 0.05, 0.08]:
        fp = make_lr_fp(C)
        p = np.zeros(len(y))
        for tr_idx, va_idx in FOLDS[SEEDS[0]]:
            p[va_idx] = fp(tr_idx, va_idx, SEEDS[0])[0]
        grid[C] = log_loss(y, p)
    LR_C = min(grid, key=grid.get)
    print('LR grid (logloss):', {k: round(v, 5) for k, v in grid.items()}, '-> C =', LR_C)
    run_model('logreg_cross', make_lr_fp(LR_C))

LR grid (logloss): {0.02: 0.28011, 0.035: 0.28077, 0.05: 0.28169, 0.08: 0.28339} -> C = 0.02
logreg_cross   CV error 0.11193 ± 0.00058 | logloss 0.28021 | 0.7 min


## Model 5 — Neural net with entity embeddings (PyTorch, GPU)
Each category gets a learned vector; an MLP on top. Different inductive bias from trees and from the linear model, which is exactly what a blend needs.

In [8]:
import torch
import torch.nn as nn

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EMB_COLS = CATS2
NN_EPOCHS, NN_BS = (3 if FAST else 18), 256

categories = {c: pd.Categorical(pd.concat([X2[c], X2_test[c]])).categories for c in EMB_COLS}

def nn_encode(df):
    cat = np.column_stack([pd.Categorical(df[c], categories=categories[c]).codes for c in EMB_COLS]).astype(np.int64)
    num = np.column_stack([df['age_of_respondent'], df['household_size']] +
                          [np.log1p(df[c]) for c in CNTS]).astype(np.float32)
    return cat, num

N_cat, N_num = nn_encode(X2)
T_cat, T_num = nn_encode(X2_test)
mu, sd = np.vstack([N_num, T_num]).mean(0), np.vstack([N_num, T_num]).std(0) + 1e-6
N_num, T_num = (N_num - mu) / sd, (T_num - mu) / sd

class EmbNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(len(categories[c]), min(16, (len(categories[c]) + 1) // 2 + 1))
                                   for c in EMB_COLS])
        d = sum(e.embedding_dim for e in self.embs) + N_num.shape[1]
        self.mlp = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(d, 256), nn.BatchNorm1d(256), nn.SiLU(), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.BatchNorm1d(64), nn.SiLU(), nn.Dropout(0.2),
            nn.Linear(64, 1))

    def forward(self, c, x):
        h = torch.cat([e(c[:, i]) for i, e in enumerate(self.embs)] + [x], dim=1)
        return self.mlp(h).squeeze(1)

def to_t(a):
    return torch.from_numpy(a).to(DEVICE)

@torch.no_grad()
def nn_predict(model, cat, num):
    model.eval()
    return torch.sigmoid(model(to_t(cat), to_t(num))).cpu().numpy()

def nn_fp(tr_idx, va_idx, seed):
    torch.manual_seed(seed)
    model = EmbNet().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    steps = len(tr_idx) // NN_BS
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, total_steps=NN_EPOCHS * steps)
    c_tr, x_tr = to_t(N_cat[tr_idx]), to_t(N_num[tr_idx])
    y_tr = torch.tensor(y[tr_idx], dtype=torch.float32, device=DEVICE)
    loss_fn = nn.BCEWithLogitsLoss()
    g = torch.Generator(device='cpu').manual_seed(seed)
    for _ in range(NN_EPOCHS):
        model.train()
        perm = torch.randperm(len(tr_idx), generator=g).to(DEVICE)
        for b in range(steps):                       # drops the last partial batch (BatchNorm-safe)
            i = perm[b * NN_BS:(b + 1) * NN_BS]
            opt.zero_grad()
            loss_fn(model(c_tr[i], x_tr[i]), y_tr[i]).backward()
            opt.step()
            sched.step()
    return nn_predict(model, N_cat[va_idx], N_num[va_idx]), nn_predict(model, T_cat, T_num)

if RUN['nn']:
    print('NN device:', DEVICE)
    run_model('embed_nn', nn_fp)

NN device: cuda
embed_nn       CV error 0.11327 ± 0.00045 | logloss 0.28452 | 1.9 min


## Combine: weighted blend vs. per-country stacker
Both combiners are judged on predictions they were **not** fitted on (5-fold CV over the OOF matrix), so the comparison is fair.

In [9]:
from scipy.optimize import minimize

names = list(OOF)
P = np.column_stack([OOF[n] for n in names])
T = np.column_stack([TEST[n] for n in names])
META_FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=2024).split(P, y))

def fit_weights(Pm, ym):
    def loss(w):
        w = np.abs(w) / np.abs(w).sum()
        return log_loss(ym, np.clip(Pm @ w, 1e-6, 1 - 1e-6))
    r = minimize(loss, np.ones(Pm.shape[1]) / Pm.shape[1], method='Nelder-Mead',
                 options=dict(maxiter=4000, xatol=1e-5, fatol=1e-7))
    return np.abs(r.x) / np.abs(r.x).sum()

def logit(p):
    p = np.clip(p, 1e-5, 1 - 1e-5)
    return np.log(p / (1 - p))

countries = sorted(train['country'].unique())
def stack_features(Pm, ctry):
    Z = logit(Pm)
    C = np.column_stack([(ctry == c).astype(float) for c in countries])
    inter = np.column_stack([Z[:, j] * C[:, k] for j in range(Z.shape[1]) for k in range(C.shape[1])])
    return np.hstack([Z, C, inter])

S, S_test = stack_features(P, train['country'].values), stack_features(T, test['country'].values)

# honest evaluation of both combiners
blend_oof, stack_oof = np.zeros(len(y)), np.zeros(len(y))
for tr, va in META_FOLDS:
    blend_oof[va] = P[va] @ fit_weights(P[tr], y[tr])
    stack_oof[va] = LogisticRegression(C=0.5, max_iter=2000).fit(S[tr], y[tr]).predict_proba(S[va])[:, 1]

# final fits on all OOF rows
W = fit_weights(P, y)
blend_test = T @ W
stacker = LogisticRegression(C=0.5, max_iter=2000).fit(S, y)
stack_test = stacker.predict_proba(S_test)[:, 1]

print('blend weights:', dict(zip(names, W.round(3))))
COMBINERS = {'blend': (blend_oof, blend_test), 'stack': (stack_oof, stack_test)}
for k, (o, _) in COMBINERS.items():
    print(f'{k:6s} held-out CV error {error_rate(o):.5f} | logloss {log_loss(y, o):.5f}')
RECOMMENDED = min(COMBINERS, key=lambda k: log_loss(y, COMBINERS[k][0]))
print('recommended combiner (lower logloss):', RECOMMENDED)

blend weights: {'catboost': np.float64(0.346), 'catboost_v2': np.float64(0.212), 'lightgbm': np.float64(0.204), 'logreg_cross': np.float64(0.174), 'embed_nn': np.float64(0.064)}
blend  held-out CV error 0.11210 | logloss 0.27754
stack  held-out CV error 0.11244 | logloss 0.27832
recommended combiner (lower logloss): blend


## Cutoff check
Only moves away from 0.5 if the gain beats a noise floor of 0.001 (~24 training rows).

In [10]:
grid = np.round(np.arange(0.40, 0.601, 0.01), 2)
THRESH = {}
for k, (o, _) in COMBINERS.items():
    errs = np.array([error_rate(o, t) for t in grid])
    noise = max(np.mean([SCORES[n]['cv_error_std'] for n in names]), 0.001)
    THRESH[k] = grid[errs.argmin()] if errs.min() < error_rate(o) - noise else 0.5
    print(f'{k}: best cutoff {grid[errs.argmin()]} (err {errs.min():.5f}) -> using {THRESH[k]}')

blend: best cutoff 0.48 (err 0.11159) -> using 0.5
stack: best cutoff 0.44 (err 0.11167) -> using 0.5


## Leaderboard of models

In [11]:
board = pd.DataFrame(SCORES).T
for k, (o, _) in COMBINERS.items():
    board.loc[k.upper()] = [error_rate(o, THRESH[k]), np.nan, log_loss(y, o), np.nan]
board['vs_public_best'] = board['cv_error'] - PREVIOUS_BEST
display(board.sort_values('logloss').round(5))
print('Note: CV error and public-LB error are on different rows, so compare models with each other, not with the LB number.')

,cv_error,cv_error_std,logloss,minutes,vs_public_best
BLEND,0.11210,NaN,0.27754,NaN,0.00294
STACK,0.11244,NaN,0.27832,NaN,0.00328
catboost_v2,0.11225,0.00040,0.27913,19.98628,0.00309
catboost,0.11148,0.00047,0.27929,15.55647,0.00232
logreg_cross,0.11193,0.00058,0.28021,0.67205,0.00277
lightgbm,0.11260,0.00067,0.28157,0.63093,0.00344
embed_nn,0.11327,0.00045,0.28452,1.85198,0.00411


Note: CV error and public-LB error are on different rows, so compare models with each other, not with the LB number.


## Write submissions

In [12]:
def save(name, probs, t=0.5):
    sub = pd.DataFrame({'unique_id': test_ids, 'bank_account': (probs > t).astype(int)})
    path = f'{OUT_DIR}/submission_v2_{name}.csv'
    sub.to_csv(path, index=False)
    print(f'{path}: {sub.bank_account.sum()} ones / {len(sub)} rows')
    return sub

subs = {k: save(k, COMBINERS[k][1], THRESH[k]) for k in COMBINERS}
save('recommended', COMBINERS[RECOMMENDED][1], THRESH[RECOMMENDED])

pd.DataFrame({'unique_id': test_ids, **{n: TEST[n] for n in names},
              'blend': blend_test, 'stack': stack_test}).to_csv(f'{OUT_DIR}/test_probabilities_v2.csv', index=False)

# How different is v2 from what we already submitted?
prev_path = find_file('submission_blend.csv')
if prev_path:
    prev = pd.read_csv(prev_path).set_index('unique_id')['bank_account']
    for k, s in subs.items():
        changed = (s.set_index('unique_id')['bank_account'] != prev.reindex(s['unique_id']).values).sum()
        print(f'{k}: {changed} rows differ from the previous submission ({prev_path})')

/kaggle/working/submission_v2_blend.csv: 755 ones / 10086 rows
/kaggle/working/submission_v2_stack.csv: 762 ones / 10086 rows
/kaggle/working/submission_v2_recommended.csv: 755 ones / 10086 rows
